# CMPSC 448 HW3: pretraining a small language model on WikiText-103

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO_URL = "https://github.com/Kevin-Coulter/small-language-model.git"  # a private repo needs https://<token>@github.com/...
!git clone -q {REPO_URL} /content/hw3
%cd /content/hw3

## Mount Google Drive

Checkpoints and logs are saved to Drive during training, so a disconnect loses nothing.

In [ ]:
import os
from google.colab import drive

drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/cmpsc448-hw3/out"
os.makedirs(OUT, exist_ok=True)

## 1. Data and tokenizer

Downloads WikiText-103, cleans it, trains the BPE tokenizer and writes the token ids. Takes about 10 minutes. The data stays on the runtime's local disk.

In [ ]:
!python prepare_data.py --vocab_size 8192

## 2. Smoke test

A 100-step run that checks training, evaluation, sampling and checkpointing before the real run. Output goes to a sibling folder, `out_smoke`.

In [ ]:
!python train.py --smoke --out_dir {OUT}

## 3. Training

`--minutes` is the time budget: after 50 steps the script measures its speed and sets the step count and LR schedule to fit.

After a disconnect, rerun the clone, Drive and data cells, then resume with the same flags as the original run.

In [ ]:
!python train.py --minutes 45 --out_dir {OUT}

In [ ]:
# !python train.py --minutes 45 --out_dir {OUT} --resume

In [ ]:
from IPython.display import Image

!python plot_loss.py --log {OUT}/log.csv --out {OUT}/loss.png
Image(f"{OUT}/loss.png")

## 4. Prompting the model

In [ ]:
!python generate.py --ckpt {OUT}/demo.pt --num_samples 2 --prompt "The Beatles were"
!python generate.py --ckpt {OUT}/demo.pt --num_samples 2 --prompt "= Pink Floyd =\n"

In [ ]:
import types
from generate import load, stream

model, tok, ck = load(f"{OUT}/demo.pt", "cuda")
opts = types.SimpleNamespace(max_new_tokens=200, temperature=0.8, top_k=50, top_p=0.95, repetition_penalty=1.1)
while True:
    prompt = input("prompt (empty to stop): ")
    if not prompt:
        break
    prompt = prompt.replace("\\n", "\n")
    print(prompt, end="")
    stream(model, tok, prompt, "cuda", opts)